# Chapter 5: Syntax and Parsing

Syntax and parsing are crucial components of NLP that focus on the structure and organization of sentences. Understanding syntax helps in deciphering the grammatical structure of sentences, which is essential for enabling machines to interpret and generate human language accurately.

**Parsing** involves breaking down sentences into their constituent parts and analyzing their grammatical relationships. This is fundamental for applications such as machine translation, sentiment analysis, and information extraction.

This notebook covers three main topics:
1. **Parts of Speech (POS) Tagging** -- identifying grammatical categories of words
2. **Named Entity Recognition (NER)** -- identifying and classifying key information (people, organizations, locations)
3. **Dependency Parsing** -- mapping out dependencies between words in a sentence

## Setup

Install and import the required libraries.

In [ ]:
# Install required packages (uncomment if needed)
# !pip install nltk spacy scikit-learn
# !python -m spacy download en_core_web_sm

In [ ]:
import nltk
import spacy

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('treebank', quiet=True)

# Load spaCy model
nlp = spacy.load('en_core_web_sm')

print('All libraries loaded successfully.')

---
## 5.1 Parts of Speech (POS) Tagging

POS tagging is the process of assigning grammatical categories (such as nouns, verbs, adjectives, and adverbs) to each word in a sentence. It is essential for understanding the syntactic structure of sentences and serves as a foundation for more advanced NLP tasks like parsing, NER, and machine translation.

### 5.1.1 Understanding POS Tags

Each word in a sentence is tagged with a label indicating its grammatical role. Common POS tags include:

| Tag | Part of Speech | Examples |
|-----|---------------|----------|
| **NN** | Noun | dog, city, happiness |
| **VB** | Verb | run, is, think |
| **JJ** | Adjective | big, blue, interesting |
| **RB** | Adverb | quickly, very, seldom |
| **PRP** | Pronoun | he, they, it |
| **IN** | Preposition | on, in, under |

### 5.1.2 Implementing POS Tagging with NLTK

The NLTK library includes pre-trained POS taggers that can be used out of the box. We tokenize text into words, then apply `pos_tag` to assign tags.

In [ ]:
from nltk import word_tokenize, pos_tag

# Sample text
text = "Natural Language Processing with Python is fascinating."

# Tokenize the text into words
tokens = word_tokenize(text)
print("Tokens:", tokens)

# Perform POS tagging
pos_tags = pos_tag(tokens)
print("\nPOS Tags:")
print(pos_tags)

**Reading the output:** Each tuple contains a word and its POS tag. For example:
- `('Natural', 'JJ')` -- adjective
- `('Language', 'NN')` -- noun
- `('Python', 'NNP')` -- proper noun
- `('is', 'VBZ')` -- verb (3rd person singular present)
- `('fascinating', 'VBG')` -- verb (gerund/present participle)

### POS Tagging with spaCy

spaCy also provides POS tagging as part of its processing pipeline. It offers both coarse-grained (`.pos_`) and fine-grained (`.tag_`) tags.

In [ ]:
doc = nlp("Natural Language Processing with Python is fascinating.")

print(f"{'Token':<15} {'POS':<8} {'Fine Tag':<10} {'Explanation'}")
print("-" * 55)
for token in doc:
    print(f"{token.text:<15} {token.pos_:<8} {token.tag_:<10} {spacy.explain(token.tag_)}")

### 5.1.3 Evaluating POS Taggers

We can evaluate POS tagger performance using standard metrics:
- **Accuracy**: proportion of words correctly tagged
- **Precision**: proportion of correctly tagged words among those identified as a specific POS
- **Recall**: proportion of actual instances of a POS that were correctly identified
- **F1 Score**: harmonic mean of precision and recall

Performance can vary based on text domain, language, and word ambiguity (e.g., "run" can be a verb or a noun).

Below we evaluate NLTK's `PerceptronTagger` against the Penn Treebank corpus.

In [ ]:
from nltk.corpus import treebank
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load the treebank corpus
test_data = treebank.tagged_sents()[3000:]
test_sentences = [[word for word, tag in sent] for sent in test_data]
gold_standard = [[tag for word, tag in sent] for sent in test_data]

# Tag the test sentences using the pre-trained PerceptronTagger
tagger = nltk.PerceptronTagger()
predicted_tags = [tagger.tag(sent) for sent in test_sentences]
predicted_tags = [[tag for word, tag in sent] for sent in predicted_tags]

# Flatten the lists to compute metrics
gold_flat = [tag for sent in gold_standard for tag in sent]
pred_flat = [tag for sent in predicted_tags for tag in sent]

# Compute evaluation metrics
accuracy = accuracy_score(gold_flat, pred_flat)
precision = precision_score(gold_flat, pred_flat, average='weighted', zero_division=0)
recall = recall_score(gold_flat, pred_flat, average='weighted', zero_division=0)
f1 = f1_score(gold_flat, pred_flat, average='weighted', zero_division=0)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

### 5.1.4 Training Custom POS Taggers

When working with domain-specific text (e.g., medical, legal), pre-trained taggers may not perform well. NLTK provides `UnigramTagger` and `BigramTagger` for training custom taggers on annotated corpora.

- **UnigramTagger**: assigns the most frequent tag for each word
- **BigramTagger**: considers the previous word's tag for better context; uses a backoff tagger when it cannot make a prediction

In [ ]:
from nltk.tag import UnigramTagger, BigramTagger
from nltk.corpus import treebank

# Split the treebank corpus into train/test
train_data = treebank.tagged_sents()[:3000]
test_data = treebank.tagged_sents()[3000:]

# Train a UnigramTagger
unigram_tagger = UnigramTagger(train_data)
unigram_acc = unigram_tagger.evaluate(test_data)
print(f"Unigram Tagger Accuracy: {unigram_acc:.4f}")

# Train a BigramTagger with UnigramTagger as backoff
bigram_tagger = BigramTagger(train_data, backoff=unigram_tagger)
bigram_acc = bigram_tagger.evaluate(test_data)
print(f"Bigram Tagger Accuracy:  {bigram_acc:.4f}")

The BigramTagger achieves higher accuracy because it considers the previous word's tag, providing additional context. The backoff mechanism ensures that if the BigramTagger cannot assign a tag, it falls back to the UnigramTagger.

**Advantages of custom POS taggers:**
- Higher accuracy on domain-specific text
- Better handling of unique vocabulary and jargon
- Improved performance on specialized tasks

### 5.1.5 Applications of POS Tagging

POS tagging is a foundational step in many NLP applications:

- **Parsing** -- understanding grammatical structure for constructing parse trees
- **Named Entity Recognition** -- distinguishing proper nouns from common nouns to identify entities
- **Sentiment Analysis** -- adjectives often carry sentiment (e.g., "happy", "sad")
- **Information Extraction** -- identifying verbs and their subjects/objects
- **Machine Translation** -- maintaining syntactic integrity during translation
- **Text-to-Speech** -- determining correct intonation and emphasis
- **Grammar Checking** -- detecting incorrect verb tenses, subject-verb agreement errors

---
## 5.2 Named Entity Recognition (NER)

NER is a subtask of information extraction that identifies and classifies named entities in unstructured text into predefined categories such as persons, organizations, locations, dates, monetary values, and more.

NER is crucial for understanding context and meaning, enabling better organization and retrieval of information in applications like question answering, information retrieval, and content categorization.

### 5.2.1 Understanding NER Categories

| Category | Label | Examples |
|----------|-------|----------|
| Person | PER / PERSON | Albert Einstein, Marie Curie |
| Organization | ORG | Google, United Nations |
| Location | LOC / GPE | Paris, Mount Everest |
| Date | DATE | January 1, 2024 |
| Money | MONEY | $1 billion, 500 euros |
| Miscellaneous | MISC | 20%, Olympics |

### 5.2.2 Implementing NER with spaCy

spaCy provides pre-trained models that perform NER out of the box. The `doc.ents` attribute contains recognized entities with their text and labels.

In [ ]:
import spacy

nlp = spacy.load('en_core_web_sm')

# Sample text
text = "Apple is looking at buying U.K. startup for 1 billion."

# Process the text
doc = nlp(text)

# Print named entities with their labels
print("Named Entities:")
print(f"{'Entity':<20} {'Label':<10} {'Description'}")
print("-" * 55)
for ent in doc.ents:
    print(f"{ent.text:<20} {ent.label_:<10} {spacy.explain(ent.label_)}")

In [ ]:
# Visualize entities inline (works in Jupyter)
from spacy import displacy

displacy.render(doc, style='ent', jupyter=True)

Let's try a more complex example with multiple entity types.

In [ ]:
text2 = "Barack Obama was born on August 4, 1961, in Honolulu, Hawaii."
doc2 = nlp(text2)

print("Named Entities:")
for ent in doc2.ents:
    print(f"  {ent.text:<25} {ent.label_:<10} {spacy.explain(ent.label_)}")

print()
displacy.render(doc2, style='ent', jupyter=True)

### 5.2.3 Evaluating NER Systems

NER evaluation uses the same core metrics as classification:

- **Precision**: of the entities the system found, how many were correct?
- **Recall**: of the entities that exist, how many did the system find?
- **F1 Score**: harmonic mean of precision and recall

Below is a simple demonstration of comparing predicted entities to ground truth using token-level labels.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Simulate token-level BIO labels for evaluation
# Sentence: "Apple is looking at buying U.K. startup for 1 billion ."
# Ground truth labels
true_labels =      ["B-ORG", "O", "O", "O", "O", "B-GPE", "O", "O", "B-MONEY", "I-MONEY", "O"]
# Predicted labels (from our NER system)
predicted_labels = ["B-ORG", "O", "O", "O", "O", "B-GPE", "O", "O", "B-MONEY", "I-MONEY", "O"]

# Calculate metrics (excluding 'O' tags for meaningful evaluation)
labels_of_interest = [l for l in set(true_labels) if l != 'O']

precision = precision_score(true_labels, predicted_labels, average='weighted', labels=labels_of_interest, zero_division=0)
recall = recall_score(true_labels, predicted_labels, average='weighted', labels=labels_of_interest, zero_division=0)
f1 = f1_score(true_labels, predicted_labels, average='weighted', labels=labels_of_interest, zero_division=0)

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

### 5.2.4 Training Custom NER Models

When pre-trained models do not cover your domain-specific entities, you can train a custom NER model with spaCy. The steps are:

1. Create a blank model
2. Add an NER component with custom labels
3. Prepare annotated training data with entity character offsets
4. Train iteratively using `nlp.update()`

Below we train a model to recognize a custom entity type `GADGET`.

In [ ]:
import spacy
from spacy.training import Example
from spacy.util import minibatch, compounding
import random

# Create a blank English model
nlp_custom = spacy.blank("en")

# Add NER component
ner = nlp_custom.add_pipe("ner")
ner.add_label("GADGET")

# Training data: (text, {"entities": [(start, end, label)]})
TRAIN_DATA = [
    ("Apple is releasing a new iPhone.", {"entities": [(26, 32, "GADGET")]}),
    ("The new iPad Pro is amazing.", {"entities": [(8, 16, "GADGET")]}),
    ("Samsung launched the Galaxy S24.", {"entities": [(20, 30, "GADGET")]}),
    ("I love my new MacBook Air.", {"entities": [(14, 25, "GADGET")]}),
    ("The Pixel 8 has a great camera.", {"entities": [(4, 11, "GADGET")]}),
]

# Train the NER model
optimizer = nlp_custom.begin_training()
for epoch in range(20):
    random.shuffle(TRAIN_DATA)
    losses = {}
    for text, annotations in TRAIN_DATA:
        doc = nlp_custom.make_doc(text)
        example = Example.from_dict(doc, annotations)
        nlp_custom.update([example], drop=0.5, losses=losses, sgd=optimizer)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:>2}, Loss: {losses['ner']:.4f}")

# Test the trained model
test_doc = nlp_custom("I just bought a new iPhone.")
print("\nPredicted Entities:", [(ent.text, ent.label_) for ent in test_doc.ents])

**Note:** This is a minimal example for illustration. In practice, you would need significantly more training data and should use spaCy's config-based training system (`spacy train`) for production models.

### 5.2.5 Applications of NER

- **Information Retrieval** -- filtering and ranking documents by significant entities
- **Question Answering** -- identifying entities that answer specific questions
- **Content Categorization** -- tagging articles by person, organization, or location
- **Customer Support** -- identifying products, services, and issues in user queries

---
## 5.3 Dependency Parsing

Dependency parsing is a syntactic analysis task that identifies the grammatical structure of a sentence by establishing relationships (dependencies) between words. Each dependency connects a **head** (governor) and a **dependent** (modifier).

### 5.3.1 Understanding Dependency Parsing

The syntactic structure is represented as a **dependency tree** where:
- **Nodes** represent words
- **Edges** represent dependency relations (directed from head to dependent)

For example, in "The cat sat on the mat":
- `The` (det) --> `cat`
- `cat` (nsubj) --> `sat`
- `sat` (ROOT) -- root verb
- `on` (prep) --> `sat`
- `the` (det) --> `mat`
- `mat` (pobj) --> `on`

### 5.3.2 Dependency Parsing with spaCy

spaCy provides pre-trained models that can parse dependency structure and label each relation.

In [ ]:
import spacy
from spacy import displacy

nlp = spacy.load('en_core_web_sm')

text = "The cat sat on the mat."
doc = nlp(text)

# Print dependency parsing results
print("Dependency Parsing:")
print(f"{'Token':<10} {'Dep Label':<12} {'Head':<10} {'Head POS':<10} {'Children'}")
print("-" * 65)
for token in doc:
    children = [child.text for child in token.children]
    print(f"{token.text:<10} {token.dep_:<12} {token.head.text:<10} {token.head.pos_:<10} {children}")

In [ ]:
# Visualize the dependency tree
displacy.render(doc, style='dep', jupyter=True, options={'distance': 100})

Let's try a more complex sentence to see richer dependency structure.

In [ ]:
doc3 = nlp("She enjoys reading books about artificial intelligence.")

print("Dependency Parsing:")
for token in doc3:
    print(f"  {token.text:<20} --({token.dep_})--> {token.head.text}")

print()
displacy.render(doc3, style='dep', jupyter=True, options={'distance': 90})

### Navigating the Dependency Tree

spaCy lets you traverse the tree programmatically to extract subtrees, find subjects/objects, etc.

In [ ]:
doc4 = nlp("The quick brown fox jumps over the lazy dog.")

# Find the root verb
root = [token for token in doc4 if token.dep_ == 'ROOT'][0]
print(f"Root verb: {root.text}")

# Find the subject
subject = [child for child in root.children if child.dep_ == 'nsubj']
if subject:
    subj = subject[0]
    print(f"Subject: {subj.text}")
    print(f"Subject subtree: {' '.join([t.text for t in subj.subtree])}")

# Find prepositional phrases
preps = [child for child in root.children if child.dep_ == 'prep']
for prep in preps:
    print(f"Prep phrase: {' '.join([t.text for t in prep.subtree])}")

### 5.3.3 Evaluating Dependency Parsers

Two standard metrics for evaluating dependency parsers:

- **Unlabeled Attachment Score (UAS)**: percentage of words assigned the correct head, regardless of the dependency label
- **Labeled Attachment Score (LAS)**: percentage of words assigned both the correct head AND the correct dependency label (stricter)

Below we compute UAS and LAS manually on a small example to illustrate the concept.

In [ ]:
# Manual evaluation example: "The cat sat on the mat."
# Gold standard: (token, head_index, dep_label)
gold = [
    ("The",  1, "det"),
    ("cat",  2, "nsubj"),
    ("sat",  2, "ROOT"),    # root points to itself
    ("on",   2, "prep"),
    ("the",  5, "det"),
    ("mat",  3, "pobj"),
    (".",    2, "punct"),
]

# spaCy's predictions
doc = nlp("The cat sat on the mat.")
predicted = []
for token in doc:
    head_idx = token.head.i  # index of the head token
    predicted.append((token.text, head_idx, token.dep_))

# Compute UAS and LAS
correct_head = 0
correct_both = 0
total = len(gold)

for g, p in zip(gold, predicted):
    if g[1] == p[1]:  # correct head
        correct_head += 1
        if g[2] == p[2]:  # correct label too
            correct_both += 1

uas = correct_head / total
las = correct_both / total

print(f"Unlabeled Attachment Score (UAS): {uas:.4f}")
print(f"Labeled Attachment Score (LAS):   {las:.4f}")

### 5.3.4 Training Custom Dependency Parsers

For domain-specific parsing, you can train a custom dependency parser with spaCy. Training data specifies the `heads` (index of the head token for each word) and `deps` (dependency labels).

Below is a simplified example to demonstrate the workflow.

In [ ]:
import spacy
from spacy.training import Example
from spacy.util import minibatch, compounding
import random

# Create a blank English model
nlp_parser = spacy.blank("en")

# Add a parser component
parser = nlp_parser.add_pipe("parser")

# Add dependency labels
for label in ["nsubj", "ROOT", "dobj", "prep", "pobj", "punct", "aux"]:
    parser.add_label(label)

# Training data: heads = index of head token, deps = dependency label
TRAIN_DATA = [
    ("She enjoys playing tennis.", {
        "heads": [1, 1, 1, 2, 1],
        "deps":  ["nsubj", "ROOT", "dobj", "dobj", "punct"]
    }),
    ("I like reading books.", {
        "heads": [1, 1, 1, 2, 1],
        "deps":  ["nsubj", "ROOT", "dobj", "dobj", "punct"]
    }),
    ("He can swim fast.", {
        "heads": [2, 2, 2, 2, 2],
        "deps":  ["nsubj", "aux", "ROOT", "dobj", "punct"]
    }),
]

# Train the parser
optimizer = nlp_parser.begin_training()
for epoch in range(20):
    random.shuffle(TRAIN_DATA)
    losses = {}
    for text, annotations in TRAIN_DATA:
        doc = nlp_parser.make_doc(text)
        example = Example.from_dict(doc, annotations)
        nlp_parser.update([example], drop=0.5, losses=losses, sgd=optimizer)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:>2}, Loss: {losses['parser']:.4f}")

# Test the trained parser
test_doc = nlp_parser("She enjoys reading books.")
print("\nParsed result:")
for token in test_doc:
    print(f"  {token.text} ({token.dep_}) --> {token.head.text}")

**Note:** This minimal example is for illustration only. Real custom parsers require much larger annotated datasets and should use spaCy's config-based training pipeline.

### 5.3.5 Applications of Dependency Parsing

- **Information Extraction** -- identifying entities and their relationships (e.g., "Obama" born in "Hawaii")
- **Machine Translation** -- maintaining syntactic integrity across languages
- **Sentiment Analysis** -- handling negation ("don't like") by understanding grammatical relationships
- **Question Answering** -- identifying subject, verb, object to match questions to answers
- **Text Summarization** -- extracting main ideas based on syntactic structure
- **Coreference Resolution** -- linking pronouns to their antecedents
- **Text Generation** -- producing grammatically correct output

---
## Practical Exercises

### Exercise 1: POS Tagging

**Task:** Perform POS tagging on the sentence: *"The quick brown fox jumps over the lazy dog."*

Use both NLTK and spaCy, and compare the outputs.

In [ ]:
from nltk import word_tokenize, pos_tag

sentence = "The quick brown fox jumps over the lazy dog."

# NLTK POS tagging
tokens = word_tokenize(sentence)
nltk_tags = pos_tag(tokens)
print("NLTK POS Tags:")
print(nltk_tags)

# spaCy POS tagging
doc = nlp(sentence)
print("\nspaCy POS Tags:")
print([(token.text, token.pos_, token.tag_) for token in doc])

### Exercise 2: Named Entity Recognition

**Task:** Perform NER on the sentence: *"Barack Obama was born on August 4, 1961, in Honolulu, Hawaii."*

Print each entity with its label and visualize using displacy.

In [ ]:
import spacy
from spacy import displacy

nlp = spacy.load('en_core_web_sm')

text = "Barack Obama was born on August 4, 1961, in Honolulu, Hawaii."
doc = nlp(text)

print("Named Entities:")
for ent in doc.ents:
    print(f"  {ent.text:<25} {ent.label_}")

displacy.render(doc, style='ent', jupyter=True)

### Exercise 3: Dependency Parsing

**Task:** Perform dependency parsing on the sentence: *"She enjoys reading books."*

Print each token's dependency relation and head, then visualize the dependency tree.

In [ ]:
text = "She enjoys reading books."
doc = nlp(text)

print("Dependency Parsing:")
for token in doc:
    print(f"  {token.text:<12} ({token.dep_:<8}) --> {token.head.text}")

print()
displacy.render(doc, style='dep', jupyter=True, options={'distance': 100})

### Exercise 4: Combined Analysis

**Task:** Given a paragraph, extract all unique named entities and their POS tags in context. This combines POS tagging and NER on real-world text.

In [ ]:
paragraph = """
Elon Musk founded SpaceX in 2002 in Hawthorne, California.
The company has launched over 200 missions and aims to reach Mars by 2030.
NASA awarded SpaceX a $2.9 billion contract for the Artemis program.
"""

doc = nlp(paragraph)

# Extract entities
print("Named Entities Found:")
print(f"{'Entity':<25} {'Label':<12} {'Description'}")
print("-" * 60)
for ent in doc.ents:
    print(f"{ent.text:<25} {ent.label_:<12} {spacy.explain(ent.label_)}")

# Show POS tags for entity tokens
print("\nPOS Tags for Entity Tokens:")
for ent in doc.ents:
    tokens_info = [(t.text, t.pos_) for t in ent]
    print(f"  {ent.text:<25} -> {tokens_info}")

### Exercise 5: Extracting Subject-Verb-Object Triples

**Task:** Use dependency parsing to extract subject-verb-object (SVO) triples from sentences. This is a practical application of dependency parsing for information extraction.

In [ ]:
def extract_svo(doc):
    """Extract subject-verb-object triples from a spaCy Doc."""
    triples = []
    for token in doc:
        if token.dep_ == 'ROOT':
            verb = token
            subjects = [child for child in verb.children if child.dep_ in ('nsubj', 'nsubjpass')]
            objects = [child for child in verb.children if child.dep_ in ('dobj', 'attr', 'prep')]
            for subj in subjects:
                subj_phrase = ' '.join([t.text for t in subj.subtree])
                for obj in objects:
                    obj_phrase = ' '.join([t.text for t in obj.subtree])
                    triples.append((subj_phrase, verb.text, obj_phrase))
    return triples


sentences = [
    "The cat sat on the mat.",
    "Alice wrote a beautiful poem.",
    "The engineer built a complex bridge.",
]

for sent in sentences:
    doc = nlp(sent)
    triples = extract_svo(doc)
    print(f"Sentence: {sent}")
    for subj, verb, obj in triples:
        print(f"  Subject: {subj}  |  Verb: {verb}  |  Object: {obj}")
    print()

---
## Chapter Summary

In this chapter we covered three foundational techniques for syntactic analysis in NLP:

1. **POS Tagging** -- Assigns grammatical categories to words. We implemented taggers with both NLTK and spaCy, evaluated them with accuracy/precision/recall/F1, and trained custom taggers using `UnigramTagger` and `BigramTagger`.

2. **Named Entity Recognition (NER)** -- Identifies and classifies entities (persons, organizations, locations, etc.) in text. We used spaCy's pre-trained models, evaluated NER output, and trained a custom NER model to recognize a new entity type.

3. **Dependency Parsing** -- Maps the grammatical structure of sentences as dependency trees. We parsed sentences with spaCy, visualized dependency trees, computed UAS/LAS metrics, and demonstrated how to train a custom parser.

These techniques form the backbone of many higher-level NLP applications and are essential building blocks for tasks like information extraction, machine translation, and question answering.